In [ ]:
import pandas as pd
import numpy as np
import wandb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
import os
os.environ["WANDB_START_METHOD"] = "thread"

In [ ]:
def evaluate_model(name, pipeline, config=None):
    """Fit pipeline, evaluate on test set, log metrics to WandB."""
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    metrics = {
        "mae":  mean_absolute_error(y_test, preds),
        "rmse": mean_squared_error(y_test, preds) ** 0.5,
        "r2":   r2_score(y_test, preds),
    }
    run = wandb.init(
        project="hpsa-score-imputation",
        name=name,
        config=config or {},
        reinit=True
    )
    wandb.log(metrics)
    run.finish()
    print(f"{name:<25} MAE={metrics['mae']:.3f}  RMSE={metrics['rmse']:.3f}  R²={metrics['r2']:.3f}")
    return pipeline

In [ ]:
def evaluate_model(name, pipeline, config=None):
    """Fit pipeline, evaluate on test set, log metrics to WandB."""
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    metrics = {
        "mae":  mean_absolute_error(y_test, preds),
        "rmse": mean_squared_error(y_test, preds) ** 0.5,
        "r2":   r2_score(y_test, preds),
    }
    run = wandb.init(
        project="hpsa-score-imputation",
        name=name,
        config=config or {},
        reinit=True,
        settings=wandb.Settings(start_method="thread")  # required on Windows
    )
    wandb.log(metrics)
    run.finish()
    print(f"{name:<25} MAE={metrics['mae']:.3f}  RMSE={metrics['rmse']:.3f}  R²={metrics['r2']:.3f}")
    return pipeline

In [ ]:
trained_models = {}

# 1. Linear Regression
trained_models['linear_regression'] = evaluate_model(
    'linear_regression',
    Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    config={'model_type': 'LinearRegression'}
)

# 2. Decision Tree
trained_models['decision_tree'] = evaluate_model(
    'decision_tree',
    DecisionTreeRegressor(random_state=42),
    config={'model_type': 'DecisionTreeRegressor', 'random_state': 42}
)

# 3. Random Forest
trained_models['random_forest'] = evaluate_model(
    'random_forest',
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    config={'model_type': 'RandomForestRegressor', 'n_estimators': 100, 'random_state': 42}
)

# 4. Gradient Boosting
trained_models['gradient_boosting'] = evaluate_model(
    'gradient_boosting',
    GradientBoostingRegressor(n_estimators=100, random_state=42),
    config={'model_type': 'GradientBoostingRegressor', 'n_estimators': 100, 'random_state': 42}
)

In [ ]:
# Feature importance for tree-based models
for name in ['random_forest', 'gradient_boosting']:
    model = trained_models[name]
    # For pipelines, access the final step; for plain estimators, access directly
    estimator = model.named_steps['model'] if hasattr(model, 'named_steps') else model
    importances = pd.Series(estimator.feature_importances_, index=FEATURES)
    print(f"\n{name} — top features:")
    print(importances.sort_values(ascending=False).round(4).to_string())

In [ ]:
# Impute hpsa_score_max for hpsa_designated == 0 using all models
X_impute = impute_df[FEATURES]

for name, pipeline in trained_models.items():
    impute_df[f'hpsa_score_max_pred_{name}'] = pipeline.predict(X_impute)

pred_cols = [c for c in impute_df.columns if c.startswith('hpsa_score_max_pred')]
print(impute_df[['geoid_tract', 'geoid'] + pred_cols].head(10))
print(f"\nNull predictions: {impute_df[pred_cols].isnull().sum().sum()}")